In [11]:
# Importando as bibliotecas que vou precisar para esse projeto
import pandas as pd 
import numpy as np
from sklearn.model_selection  import  train_test_split 
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau 
from tensorflow.keras.regularizers import l2 
from tensorflow.keras import optimizers 
from scikeras.wrappers import KerasClassifier


In [ ]:
# Função de construção da arquitetura para o modelo de classificação
def create_thyroid_model(learning_rate=0.001):
    model = Sequential(name='Thyroid_classifier_optimizer')
    model.add(Input(shape=(...)))   #Definido automaticamento pelo pré-processador
    
    model.add(Dense(32, activation= 'tanh', kernel_regularizer= l2(0.01)))
    model.add(BatchNormalization())
    model.add(Dropout(0.1))
    
    model.add(Dense(16, activation='tanh', kernel_regularizer=l2(0.01)))
    model.add(BatchNormalization())
    model.add(Dropout(0.1))
    
    model.add(Dense(8, activation='tanh', kernel_regularizer=l2(0.1))) 
    
    model.add(Dense(1, activation='sigmoid', kernel_regularizer=l2(0,1)))
    optimizer = optimizers.Adam(learning_rate = learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['Recall'])
    
    return model




    

In [ ]:
# Função principal da construção da pipeline
def run_mlops_pipeline(data_path):
    print('Iniciando Ingestão de Dados...')
    data = pd.read_csv(data_path)
    
    # Tratando as colunas do meu data set com labelencoder
    label = LabelEncoder()
    
    
    # Separação de Featureas e Target
    X = data.drop('Recurred', axis=1).values    
    y = data['Recurred'].values 
    
    x_train, x_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state = 42)
    
    early_stop = EarlyStopping(monitor='val_Recall', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    
    # Envelopando o modelo keras para o Scikit-Learn
    
    keras_estimator = KerasClassifier(
        model = create_thyroid_model,
        epochs=80,
        batch_size=64,
        validation_split=0.1,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    print('Construindo Pípeline')
    production_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', keras_estimator)
    ])
    
    print('Inciiando o Treinamento...')
    # Treinado a pipeline (ela escalona os dados e depois treina a redeu neural )
    production_pipeline.fit(x_train,y_train)
    
    print('Avaliando o Modelo...')
    # Com o uso do pipeline ele já escalona o X antes de fazer as previsões 
    y_pred_proba = production_pipeline.predict_proba(x_test)[:,1]
    y_pred = np.array([1 if p> 0.5  else 0 for p in y_pred_proba])
    
    print('-' * 25, 'Resultados em Produção', '-' * 25)   
    print(classification_report(y_test,y_pred, target_names = ['Recuperação', 'Recorrência']))
    
    return production_pipeline


In [10]:
pipeline_final = run_mlops_pipeline(r'C:\Users\cassio.sferreira\Documents\Git\Thyroid_Cancer\Thyroid_Diff.csv')
    

Iniciando Ingestão de Dados...


TypeError: only integer scalar arrays can be converted to a scalar index